# Data Cleaning for XRP 🧹

### Step 1: Loading and Initial Exploration

**Why are we doing this?** Before cleaning, we need to understand the structure, data types, and identify if there are any missing values.

In [1]:
import pandas as pd

# Load the file
df = pd.read_csv('items/XRP_Data.csv')

# Display the first few rows and technical info
print("First rows:")
display(df.head())

print("\nDataFrame Information:")
df.info()

First rows:


,Date,Ticker,Currency,Open,High,Low,Close,Volume
0,2017-11-09,XRP-USD,USD,0.217911,0.221791,0.214866,0.217488,147916992
1,2017-11-10,XRP-USD,USD,0.218256,0.219068,0.205260,0.206483,141032992
2,2017-11-11,XRP-USD,USD,0.205948,0.214456,0.205459,0.210430,134503008
3,2017-11-12,XRP-USD,USD,0.210214,0.210214,0.195389,0.197339,251175008
4,2017-11-13,XRP-USD,USD,0.197472,0.204081,0.197456,0.203442,132567000



DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      3044 non-null   object 
 1   Ticker    3044 non-null   object 
 2   Currency  3044 non-null   object 
 3   Open      3044 non-null   float64
 4   High      3044 non-null   float64
 5   Low       3044 non-null   float64
 6   Close     3044 non-null   float64
 7   Volume    3044 non-null   int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 190.4+ KB


### Step 2: Date Format Conversion

**Why are we doing this?** By default, dates are loaded as "objects" (strings). To perform time series analysis, calculate returns, or create plots by year/month, we need Python to recognize them as datetime objects.

In [2]:
# Convert the Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Verify the change
print(f"New data type for 'Date': {df['Date'].dtype}")

New data type for 'Date': datetime64[ns]


### Step 3: Handling Missing Values

**Why are we doing this?** Yahoo Finance sometimes has "gaps" on days where there was no record or the API failed. Null values can break mathematical algorithms.

In [3]:
# Count null values per column
print("Missing values found:")
print(df.isnull().sum())

# If nulls are found, the best practice in finance is 'Forward Fill' 
# (filling with the previous day's price) instead of deleting the row.
df = df.ffill()

Missing values found:
Date        0
Ticker      0
Currency    0
Open        0
High        0
Low         0
Close       0
Volume      0
dtype: int64


### Step 4: Detection and Removal of Duplicates

**Why are we doing this?** Duplicate records bias averages and total transaction volume, artificially inflating metrics.

In [4]:
# Search for identical rows
duplicates = df.duplicated().sum()
print(f"Duplicate records: {duplicates}")

# Remove if they exist (keeping the first occurrence)
if duplicates > 0:
    df = df.drop_duplicates()

Duplicate records: 0


### Step 5: Chronological Sorting and Indexing

**Why are we doing this?** In trading, the order of factors does matter. To calculate daily variations (percentage changes), data must be sorted from oldest to newest. Setting the date as the index facilitates searching and filtering.

In [5]:
# Sort by date just in case they arrived out of order
df = df.sort_values(by='Date')

# Set the date as the index
df.set_index('Date', inplace=True)

print("Data ready and sorted:")
display(df.head())

Data ready and sorted:


,Ticker,Currency,Open,High,Low,Close,Volume
Date,,,,,,,
2017-11-09,XRP-USD,USD,0.217911,0.221791,0.214866,0.217488,147916992
2017-11-10,XRP-USD,USD,0.218256,0.219068,0.205260,0.206483,141032992
2017-11-11,XRP-USD,USD,0.205948,0.214456,0.205459,0.210430,134503008
2017-11-12,XRP-USD,USD,0.210214,0.210214,0.195389,0.197339,251175008
2017-11-13,XRP-USD,USD,0.197472,0.204081,0.197456,0.203442,132567000


### Step 6: Consistency Verification (Outliers)

**Why are we doing this?** Sometimes there are decimal errors (e.g., a price that says $100 when XRP is worth $1). A quick check of the maximum and minimum values helps us detect loading errors.

In [6]:
# Statistical summary
# We look for values that don't make sense (e.g., a Low of 0 or a negative High)
display(df.describe())

,Open,High,Low,Close,Volume
count,3044.000000,3044.000000,3044.000000,3044.000000,3.044000e+03
mean,0.810512,0.838788,0.780777,0.810917,2.883614e+09
std,0.745611,0.772514,0.718544,0.745628,3.871341e+09
min,0.140524,0.146911,0.115093,0.139635,1.002940e+08
25%,0.334478,0.345358,0.324965,0.334992,9.533236e+08
50%,0.516728,0.527266,0.501907,0.517056,1.625303e+09
75%,0.859550,0.893341,0.823169,0.860899,3.192528e+09
max,3.555637,3.841940,3.430836,3.555765,5.172338e+10


### Step 7: Logical Integrity Audit and Non-Negative Values

**Why are we doing this?** In financial analysis, there are unbreakable physical rules. For example, the lowest price of a day (Low) can never be higher than the highest (High). Also, in a real market, prices and transaction volume cannot be negative or zero; if they were, we would be facing "noise" errors in data capture that would ruin any predictive model.

**Validations included:**

* **Price Hierarchy:** Verifying that $Low \leq \{Open, Close\} \leq High$.
* **Positivity:** Ensuring that $Price > 0$ and $Volume \geq 0$.
* **Type Integrity:** Confirming there are no non-numeric values in critical columns.

In [7]:
# 1. Define columns of interest
price_cols = ['Open', 'High', 'Low', 'Close']

# 2. Check for Negative or Zero Prices
# We check if there is any value less than or equal to 0 in price columns
invalid_prices = df[(df[price_cols] <= 0).any(axis=1)]

print(f"Records with invalid prices: {len(invalid_prices)}")
if len(invalid_prices) > 0:
    print("Review the following dates:", invalid_prices.index.tolist())

# 3. Check for Negative Volume
invalid_volume = df[df['Volume'] < 0]

print(f"Records with negative volume: {len(invalid_volume)}")

# 4. Check Logical Hierarchy (High and Low)
# High must be the highest price of the day
high_error = df[(df['High'] < df['Open']) | (df['High'] < df['Close']) | (df['High'] < df['Low'])]

# Low must be the lowest price of the day
low_error = df[(df['Low'] > df['Open']) | (df['Low'] > df['Close']) | (df['Low'] > df['High'])]

print(f"Inconsistencies in Maximum price (High): {len(high_error)}")
print(f"Inconsistencies in Minimum price (Low): {len(low_error)}")

# 5. Final audit conclusion
if len(invalid_prices) == 0 and len(invalid_volume) == 0 and len(high_error) == 0 and len(low_error) == 0:
    print("\n✅ SUCCESS: The data has passed all logical security tests.")
else:
    print("\n❌ WARNING: Anomalies were detected in the dataset.")

Records with invalid prices: 0
Records with negative volume: 0
Inconsistencies in Maximum price (High): 0
Inconsistencies in Minimum price (Low): 0

✅ SUCCESS: The data has passed all logical security tests.
